# 🛡️ 보험 상담 AI 에이전트 (Google Colab)

**GPT-4o + 보험다모아 엑셀 데이터 + 실시간 웹 검색** 기반 보험 상담 챗봇

### 실행 순서
1. **셀 1**: 패키지 설치
2. **셀 2**: 코드 클론
3. **셀 3**: API 키 설정
4. **셀 4**: (선택) 엑셀 데이터 업로드
5. **셀 5**: 서버 실행 → 생성된 URL로 접속

> **필요한 것**: OpenAI API 키, ngrok 계정(무료) — https://dashboard.ngrok.com/signup


## 셀 1 — 패키지 설치

In [ ]:
# 핵심 패키지
!pip install -q openai python-dotenv flask requests ddgs xlrd pyngrok

# ChromaDB RAG (선택 — 처음 실행 시 임베딩 모델 약 443MB 다운로드)
!pip install -q chromadb sentence-transformers

# PyTorch CPU (Colab은 GPU 버전이 기본 설치됨, 별도 설치 불필요)
print('✅ 패키지 설치 완료')

## 셀 2 — 코드 클론

In [ ]:
import os

REPO_URL = 'https://github.com/Sdapaul/insurance-agent.git'
PROJECT_DIR = '/content/insurance-agent'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
print(f'✅ 작업 디렉터리: {os.getcwd()}')
!ls -la

## 셀 3 — API 키 설정

### 방법 A: Colab Secrets (권장)
왼쪽 사이드바 🔑 아이콘 → `OPENAI_API_KEY` 추가

### 방법 B: 직접 입력 (아래 셀에서 입력)

In [ ]:
import os

# ── 방법 A: Colab Secrets에서 자동 로드 ──
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    FSS_API_KEY    = userdata.get('FSS_API_KEY') or ''
    NGROK_TOKEN    = userdata.get('NGROK_TOKEN') or ''
    print('✅ Colab Secrets에서 API 키 로드')
except Exception:
    OPENAI_API_KEY = ''
    FSS_API_KEY    = ''
    NGROK_TOKEN    = ''

# ── 방법 B: 직접 입력 (Secrets 미사용 시 여기에 입력) ──
if not OPENAI_API_KEY:
    OPENAI_API_KEY = 'sk-...'          # ← OpenAI API 키
if not NGROK_TOKEN:
    NGROK_TOKEN    = '2...'            # ← ngrok 인증 토큰 (dashboard.ngrok.com)
# FSS_API_KEY    = 'YOUR_FSS_KEY'     # ← 선택 (연금저축보험 조회 시)

# 환경변수 및 .env 파일 생성
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['FSS_API_KEY']    = FSS_API_KEY

with open('/content/insurance-agent/.env', 'w') as f:
    f.write(f'OPENAI_API_KEY={OPENAI_API_KEY}\n')
    f.write(f'FSS_API_KEY={FSS_API_KEY}\n')

if not OPENAI_API_KEY or OPENAI_API_KEY.startswith('sk-...'):
    print('⚠️  OpenAI API 키를 입력하세요!')
else:
    print(f'✅ OpenAI API 키 설정 완료 ({OPENAI_API_KEY[:8]}...)')

if not NGROK_TOKEN or len(NGROK_TOKEN) < 10:
    print('⚠️  ngrok 토큰을 입력하세요! → https://dashboard.ngrok.com/get-started/your-authtoken')
else:
    print('✅ ngrok 토큰 설정 완료')

## 셀 4 — (선택) 보험다모아 엑셀 데이터 업로드

보험다모아(e-insmarket.or.kr)에서 다운로드한 `.xls` 파일을 업로드하면 실제 공시 데이터로 추천을 받을 수 있습니다.  
**업로드하지 않아도** 로컬 데이터 + 웹 검색으로 동작합니다.

In [ ]:
import os, shutil
from google.colab import files

print('엑셀 파일을 업로드하세요 (취소해도 무방)...')
try:
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = os.path.join('/content/insurance-agent', fname)
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'  ✅ 업로드: {fname} ({len(data):,} bytes)')
    if uploaded:
        # 엑셀 캐시 초기화
        cache_path = '/content/insurance-agent/data/insmarket_excel_cache.json'
        if os.path.exists(cache_path):
            os.remove(cache_path)
        print('\n✅ 엑셀 캐시 초기화 완료 — 다음 질문 시 자동 파싱됩니다.')
except Exception as e:
    print(f'업로드 건너뜀 또는 오류: {e}')

## 셀 5 — 서버 실행

실행 후 출력되는 **ngrok URL** 로 접속하세요.  
서버를 중지하려면 런타임 메뉴 → **세션 다시 시작** 또는 셀 왼쪽 ■ 버튼을 누르세요.

In [ ]:
import os, sys, threading, time
os.chdir('/content/insurance-agent')
sys.path.insert(0, '/content/insurance-agent')

# ngrok 터널 설정
from pyngrok import ngrok, conf

if NGROK_TOKEN and len(NGROK_TOKEN) > 10:
    conf.get_default().auth_token = NGROK_TOKEN

# 기존 터널 종료
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

# Flask 앱 임포트
import importlib
import web_app as _wa
importlib.reload(_wa)
app = _wa.app

# 백그라운드 스레드로 Flask 실행
PORT = 5000
def run_flask():
    app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

# ngrok 터널 오픈
try:
    tunnel = ngrok.connect(PORT)
    public_url = tunnel.public_url
    print('=' * 60)
    print(f'🌐 보험 상담 AI 접속 URL:')
    print(f'   {public_url}')
    print('=' * 60)
    print('⚠️  이 셀을 중단하면 서버가 종료됩니다.')
    print('   Colab 탭을 열어두세요.')
except Exception as e:
    print(f'❌ ngrok 오류: {e}')
    print('   ngrok 토큰을 확인하세요: https://dashboard.ngrok.com/get-started/your-authtoken')
    print(f'\n   로컬 접속(Colab 내부): http://localhost:{PORT}')

## 셀 6 — (선택) ChromaDB 벡터 DB 구축

보험 지식베이스를 벡터 DB로 변환합니다 (약 2~5분 소요).  
처음 실행 시 임베딩 모델(443MB)을 다운로드합니다.

In [ ]:
os.chdir('/content/insurance-agent')
!python scripts/build_vectorstore.py
print('✅ ChromaDB 구축 완료')

## 셀 7 — (선택) 엑셀 → 지식베이스 자동 생성

셀 4에서 엑셀 파일을 업로드했다면, 이 셀로 지식베이스에 반영할 수 있습니다.

In [ ]:
os.chdir('/content/insurance-agent')
!python scripts/build_knowledge_from_excel.py --apply --rebuild
print('✅ 지식베이스 + ChromaDB 업데이트 완료')

## 참고 — Colab 제한 사항

| 기능 | Colab 지원 | 비고 |
|------|-----------|------|
| GPT-4o 채팅 | ✅ | OpenAI API 키 필요 |
| 보험다모아 엑셀 검색 | ✅ | 셀 4에서 XLS 업로드 후 가능 |
| 실시간 웹 검색 | ✅ | DuckDuckGo (API 키 불필요) |
| FSS API (연금보험) | ✅ | FSS_API_KEY 설정 시 |
| ChromaDB RAG | ✅ | 셀 6 실행 시 |
| 신용점수 포트폴리오 탭 | ✅ | 완전 지원 |
| 보험다모아 실시간 스크래핑 | ⚠️ | CDP 모드 불가, 캐시 파일 업로드로 대체 |
| NICE/KCB 신용점수 CDP 조회 | ❌ | 로컬 Chrome 필요 |

### ngrok 무료 계정 제한
- 세션당 1개 터널, 월 1GB 데이터
- 8시간마다 URL이 변경됨 (Colab 무료 플랜 세션 제한과 동일)

### 세션 유지 팁
```javascript
// 브라우저 콘솔(F12)에서 실행하면 Colab 자동 연결 유지
function KeepAlive() {
  document.querySelector('#top-toolbar .yes-button')?.click();
  setTimeout(KeepAlive, 60000);
}
KeepAlive();
```
